# RedZone EXP-022: the world model as a timing model vs Noggin Timing v1, on one Colab L4

Trains the event-token world model (~18.5M params) on catalog-OK data only (nflverse NFL 2001-24, NBA.com cdn NBA
via shufinskiy/nba_data, seasons ending 2021-25), then reads its time head at every break stoppage (timeout, end of
period, halftime, review, injury) as the distribution of time to restart. Beside it, Timing v1's LightGBM hazard is
refit on the SAME train nights and collapsed to the same 32 log-time bins. Graded per league on the test nights with
game-bootstrap CIs: restart NLL, a 2% lower bound (split conformal on valid) and its coverage, and the mean usable
dead time. No Drive, no secrets, no login; locked and scorecard nights are never downloaded.

Runtime > Change runtime type > **L4 GPU**, then Runtime > **Run all**. About 2 h (~10 CU): data ~15 min, training
60 min, tree refit + grading ~30 min. If the runtime restarts but keeps its disk, Run all again: it resumes from the
30-min checkpoint. The last cell prints grades.json and the CU spent.

In [ ]:
PIN = "dace4eff7053acb7eadb4c347f3779ce214070d6"   # kaggle-inputs commit: factory code pack
REPO = "BouncySlime1215/redzone-everything"
LEAGUES = "nfl,nba"         # plan A (catalog-OK): nflverse NFL, NBA.com cdn NBA; hoopR = nba_espn, gated
TRAIN_MIN = 60              # training wall clock; data ~15 min + trees and grading ~30 min on top
D, LAYERS, HEADS = 384, 8, 6   # ~18.5M params (EXP-103 recipe)
CU_PER_H = {"A100": 13.0, "L4": 5.0, "T4": 2.0}

In [ ]:
import json, os, subprocess, sys, tarfile, time, urllib.request
T_START = time.time()
import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU: Runtime > Change runtime type > L4 GPU, then Run all again")
GPU = torch.cuda.get_device_name(0)
RATE = next((v for k, v in CU_PER_H.items() if k in GPU), None)
print("GPU:", GPU, "| CU/h:", RATE)
RUN = "/content/wmrun"
CODE = f"{RUN}/code"
if not os.path.exists(f"{CODE}/rz/scripts/world_model_pilot.py"):
    os.makedirs(CODE, exist_ok=True)
    for i, ref in enumerate((PIN, PIN[:12], PIN, PIN[:12])):     # the raw CDN can lag a fresh push for a minute
        url = f"https://raw.githubusercontent.com/{REPO}/{ref}/factory/rz_factory_code.tar.gz"
        try:
            print("downloading", url)
            urllib.request.urlretrieve(url, f"{RUN}/code.tar.gz")
            break
        except Exception as e:
            print(" ", type(e).__name__, e)
            time.sleep(30 * i)
    else:
        raise SystemExit("could not download the code pack; Run all again in a minute")
    with tarfile.open(f"{RUN}/code.tar.gz") as t:
        t.extractall(CODE, filter="data")
print("code:", sorted(os.listdir(f"{CODE}/rz")))

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm"], check=False)
log_path = f"{RUN}/exp022_{time.strftime('%Y%m%d-%H%M')}.log"
steps = [[f"{CODE}/rz/scripts/world_model_pilot.py", "--out", f"{RUN}/wm", "--leagues", LEAGUES, "--minutes",
          str(TRAIN_MIN), "--d", str(D), "--layers", str(LAYERS), "--heads", str(HEADS), "--exp", "EXP-022"],
         [f"{CODE}/rz/scripts/exp022_tpp.py", "--wm", f"{RUN}/wm", "--leagues", LEAGUES, "--out", f"{RUN}/tpp"]]
with open(log_path, "w") as log:
    for cmd in steps:
        p = subprocess.Popen([sys.executable, "-u", *cmd], cwd=RUN, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
        for line in p.stdout:
            print(line, end="")
            log.write(line)
        rc = p.wait()
        print(cmd[0].rsplit("/", 1)[-1], "exit code:", rc)
        if rc:
            break

In [ ]:
wall_h = (time.time() - T_START) / 3600
cu = round(wall_h * RATE, 1) if RATE else None
gp = f"{RUN}/tpp/grades.json"
if os.path.exists(gp):
    g = json.load(open(gp))
    g["colab"] = {"gpu": GPU, "wall_h": round(wall_h, 2), "cu_spent": cu, "pin": PIN}
    json.dump(g, open(gp, "w"), indent=1)
    print("===== grades.json =====")
    print(json.dumps(g, indent=1))
else:
    print("no grades.json; see", log_path)
print(f"compute: {wall_h:.2f} h on {GPU} = ~{cu} CU")
try:
    from google.colab import files
    files.download(gp)
except Exception as e:
    print("grades.json is at", gp, f"({type(e).__name__})")